# VERITAS 05 — Retrieval from scratch: chunking, BM25, dense, hybrid, rerank

**Phases 5–6 of 16.**

## Chunking decides what a citation can point at

VERITAS cites *claim → chunk → document span*, so a chunk must be
(1) **self-contained** — "He stepped down in March" is worthless alone;
(2) **small enough to be precise** — embedding a 2000-token page averages away
the one sentence that matters; (3) **offset-preserving** — we keep
`(start_char, end_char)` so the UI can highlight the exact span.

Strategy: recursive *structural* splitting (headings → paragraphs → sentences)
with a token budget and ~15 % overlap, plus a heading breadcrumb prepended to
each chunk. Structural boundaries correlate with semantic ones for free —
far cheaper than embedding-based semantic chunking, which costs one encoder
pass per sentence.

## BM25, derived

$$\text{score}(D,Q)=\sum_{t\in Q}\text{IDF}(t)\cdot\frac{f(t,D)(k_1+1)}{f(t,D)+k_1\left(1-b+b\frac{|D|}{\text{avgdl}}\right)}$$

* **IDF** — a rare term is discriminative. The `1 + …` smoothing keeps it
  positive; plain RSJ IDF goes negative for common terms, letting a stopword
  *subtract* score.
* **TF saturation** `f/(f+k₁)` — concave: the 1st "revenue" is evidence, the
  20th adds nothing. Raw TF lets one keyword-stuffed page dominate.
* **Length norm** `b·|D|/avgdl` — long documents otherwise accumulate matches
  by sheer size.

## Why keep BM25 when we have embeddings?

They fail **differently**. Dense retrieval fails on exact strings it never saw:
a ticker, a docket number, a surname, a version string — exactly the tokens
that identify entities. BM25 has no vocabulary problem and needs no training.
Hybrid exists to cover both failure modes, not to be fancy.

## Fusion: RRF vs weighted sum

BM25 scores are unbounded (0–40); cosine lives in [−1,1]. Adding them directly
lets BM25 silently dominate.
* **RRF** `Σ w/(60+rank)` uses only ranks → immune to scale, no tuning.
* **Weighted min-max sum** preserves *margins* (the gap between rank 1 and 2),
  which the evidence-sufficiency check downstream actually reads.

## Reranking: bi-encoder → cross-encoder

A bi-encoder embeds query and document **independently** — that is what makes
it fast (documents embedded offline) and also its ceiling: the document vector
is computed without ever seeing the query. A cross-encoder concatenates them so
every query token attends to every document token, at `O(candidates)` forward
passes. Hence two stages: retrieve 50 cheaply, rerank 50 expensively, keep 8.

In [ ]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from veritas.tokenizer.bpe import BPETokenizer
from veritas.model.transformer import VeritasLM, ModelConfig
from veritas.rag.chunking import chunk_document, chunk_table
from veritas.rag.bm25 import BM25, tokenize
from veritas.rag.embeddings import Embedder, VectorIndex, info_nce_loss, train_embedder
from veritas.rag.hybrid import HybridRetriever, FusionWeights, rrf_fuse, weighted_fuse
from veritas.rag.reranker import CrossEncoder, train_reranker, pairwise_rank_loss

tok = BPETokenizer.load(ROOT/'checkpoints'/'tokenizer.json')
ck = ROOT/'checkpoints'/'sft.pt'
if not ck.exists(): ck = ROOT/'checkpoints'/'best.pt'
model = VeritasLM.load(ck, DEVICE) if ck.exists() else VeritasLM(
    ModelConfig(vocab_size=tok.vocab_size, d_model=384, n_layers=8, n_heads=8,
                n_kv_heads=2, max_seq_len=256)).to(DEVICE)
model.eval(); print('model from', ck.name if ck.exists() else '(random init — run 03/04 first)')

## Chunking with offsets preserved

In [ ]:
doc = '''# Acme Industries — Annual Report 2026

## Leadership
Marcus Lund was appointed chief executive effective February 2026, succeeding
Priya Raman. The board confirmed the appointment at its January meeting.

## Financial results
Revenue for the year was 1.42 billion euros, up 12% year on year. Operating
margin improved to 14.1%. The company opened 15 new offices during the period.

## Outlook
Management expects continued expansion in Asia-Pacific markets during 2027.
'''
chunks = chunk_document(doc, 'acme_ar_2026', {'title':'Acme Annual Report 2026','tier':1,
                        'date':'2026-03-01','entity':'Acme Industries'}, target_tokens=60)
for c in chunks:
    print(f'[{c.chunk_id}] heading={c.heading_path!r} chars {c.start_char}-{c.end_char}')
    print(f'   {c.text[:90]}...')
print('\nWhat gets EMBEDDED (breadcrumb restores the referent):')
print(' ', chunks[1].contextualized[:130])

In [ ]:
# Tables need row-level linearisation or the column->value binding is lost.
rows = [['Year','Revenue','Margin'], ['2024','1.10B','11.2%'], ['2025','1.27B','12.8%'], ['2026','1.42B','14.1%']]
for c in chunk_table(rows, 'acme_fin'):
    print(f'[{c.chunk_id}] {c.text}')
print('\nFlat-text chunking would yield "2026 1.42B 14.1%" — no way to know which is revenue.')

## Build the corpus and both indices

In [ ]:
from veritas.eval.benchmark import build_seed_benchmark, expand_synthetic
bench = expand_synthetic(build_seed_benchmark(), 4)
corpus, metadata = {}, {}
for it in bench.items:
    for d in it.docs:
        if d.doc_id in corpus: continue
        corpus[d.doc_id] = d.text
        metadata[d.doc_id] = {'source':d.source,'tier':d.tier,'date':d.date,
                              'entity':d.entity or it.entity,'valid_from':d.valid_from,'valid_to':d.valid_to}
print(f'{len(corpus)} chunks')

bm25 = BM25()
for cid, text in corpus.items(): bm25.add(cid, text)
bm25.finalize()
print(f'BM25: {len(bm25.postings):,} terms | avgdl {bm25.avgdl:.1f}')

embedder = Embedder(model, tok, max_len=192, device=DEVICE)
t0=time.time(); vecs = embedder.encode(list(corpus.values()))
vec = VectorIndex(embedder.dim); vec.add(list(corpus), vecs)
print(f'dense: {vecs.shape} in {time.time()-t0:.1f}s | L2-normalised: {np.linalg.norm(vecs[0]):.4f}')

### Where each retriever fails

Run both on the same queries. The pattern to look for: BM25 wins on exact rare
strings, dense wins on paraphrase. Neither wins both — which is the argument
for hybrid, stated as evidence rather than assertion.

In [ ]:
queries = ['Who is the current CEO of Acme Industries?',
           'Marcus Lund',                      # exact rare string -> BM25 should win
           'who runs the company these days']  # paraphrase, no keywords -> dense should win
for q in queries:
    s = [d for d,_ in bm25.search(q, k=3)]
    d_ = [h.doc_id for h in vec.search(embedder.encode_one(q), k=3)]
    print(f'\nQ: {q}')
    print('  BM25 :', s)
    print('  dense:', d_)
    print('  overlap:', len(set(s)&set(d_)), 'of 3')

## Contrastive training of the embedder (InfoNCE)

$$L=-\log\frac{\exp(s(q,d^+)/\tau)}{\sum_j \exp(s(q,d_j)/\tau)}$$

Every other document in the batch is a negative, so a batch of `B` gives `B−1`
free negatives — the reason contrastive retrieval training wants large batches.
`τ≈0.05` sharpens the softmax: too high and the gradient is flat, too low and
it fixates on the single hardest negative.

In [ ]:
pairs = []
for it in bench.items:
    for gid in it.gold_docs:
        if gid in corpus: pairs.append((it.question, corpus[gid]))
print(f'{len(pairs)} (query, positive) pairs')

def recall_at(k=3):
    hit = 0; n = 0
    for it in bench.items:
        if not it.gold_docs: continue
        n += 1
        got = [h.doc_id for h in vec.search(embedder.encode_one(it.question), k=k)]
        hit += len(set(got) & set(it.gold_docs)) > 0
    return hit/max(1,n)

before = recall_at()
hist = train_embedder(model, tok, pairs, steps=150, batch_size=8, lr=5e-5, device=DEVICE, log_every=50)
embedder = Embedder(model, tok, max_len=192, device=DEVICE)
vec = VectorIndex(embedder.dim); vec.add(list(corpus), embedder.encode(list(corpus.values())))
print(f'\ndense recall@3: {before:.2f} -> {recall_at():.2f}')

## Approximate search: IVF

k-means into √N cells, probe the `nprobe` nearest. On normalised vectors,
Euclidean k-means **is** spherical k-means (minimising `‖x−c‖²` = maximising
`x·c`), so assignment is one GEMM + argmax. Below ~10⁵ vectors, exact search is
a single GEMM and is both faster *and* exact — measure before reaching for ANN.

In [ ]:
import matplotlib.pyplot as plt
q = embedder.encode_one('Who is the current CEO of Acme Industries?')
exact = [h.doc_id for h in vec.search(q, k=5)]
vec.build_ivf()
if vec.centroids is not None:
    for nprobe in (1,2,4,8):
        got = [h.doc_id for h in vec.search(q, k=5, nprobe=nprobe)]
        print(f'nprobe={nprobe}: recall vs exact = {len(set(got)&set(exact))/5:.2f}')
    print('\nThe recall/speed knob: more probes -> closer to exact, more vectors scanned.')
else:
    print(f'{len(vec.ids)} vectors — too few for IVF; exact search is already optimal here.')

## Hybrid fusion

In [ ]:
retr = HybridRetriever(vec, bm25, embedder, FusionWeights(dense=1.0, sparse=1.0,
                        freshness=0.0, authority=0.0, entity=0.0, temporal=0.0), metadata=metadata)
q = 'Who is the current CEO of Acme Industries?'
print('RRF (rank-only, scale-immune):')
for c in retr.retrieve(q, k=4, mode='rrf'): print(f'  {c.score:.4f}  {c.doc_id}')
print('\nWeighted min-max sum (preserves margins, with per-signal explanation):')
for c in retr.retrieve(q, k=4, mode='weighted'):
    parts = ' '.join(f'{k_}={v:.2f}' for k_,v in c.explain.items() if v)
    print(f'  {c.score:.4f}  {c.doc_id:22s} {parts}')

## Cross-encoder reranking

Trained with a pairwise ranking loss `−log σ(s⁺ − s⁻)`: optimising the *margin
between* a good and a bad passage is the right objective for ranking, and
pointwise regression would waste capacity calibrating absolute scores that only
ever get sorted. **Hard** negatives matter — random negatives are trivially
separable and produce no gradient.

In [ ]:
from veritas.rag.reranker import mine_hard_negatives
triples = []
for it in bench.items:
    for gid in it.gold_docs:
        if gid not in corpus: continue
        for neg in mine_hard_negatives(retr, it.question, it.gold_docs, n=2):
            if neg in corpus: triples.append((it.question, corpus[gid], corpus[neg]))
print(f'{len(triples)} (query, positive, hard-negative) triples')
ce = CrossEncoder(model, tok, max_len=256, device=DEVICE)
if triples: train_reranker(ce, triples, steps=120, batch_size=4, lr=5e-5, log_every=40)

In [ ]:
cands = [c.doc_id for c in retr.retrieve(q, k=8)]
print('before rerank:', cands[:5])
print('after  rerank:', [i for i,_ in ce.rerank(q, [corpus[c] for c in cands], cands, top_k=5)])
print('\nThe cross-encoder sees query and passage TOGETHER, so it can judge')
print('entailment — which is why the same architecture is reused as the NLI')
print('head of the claim verifier in notebook 06.')

Next: **06 — the temporal + evidence layer** (the actual novelty).